# 🤖 Agente de IA para mineração de dados em BI
**TCC – Utilização de Data Mining em Ferramentas de BI (FATEC Catanduva)**

Este notebook cria um **agente de IA** que responde perguntas de negócio em português sobre uma base real de varejo (*Online Retail*, UCI). Ele decide sozinho qual técnica de mineração de dados usar, executa o código em Python e explica o resultado.

Na definição de Russell e Norvig (2013), um agente **percebe** o ambiente e **age** para alcançar objetivos. Aqui:
- **percebe** → a pergunta do usuário e os resultados das ferramentas;
- **decide** → o modelo de linguagem (Gemini) escolhe qual ferramenta chamar e com quais parâmetros;
- **age** → executa as ferramentas: resumo de vendas, ranking, **segmentação RFM com k-means**, **previsão de churn com árvore de decisão** e exportação para o **Power BI**.

**Como usar:** rode as células em ordem (▶ ou `Ctrl+Enter`). Só a etapa 2 precisa de configuração.

## 1. Instalar a biblioteca do Gemini

In [4]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configurar a chave da API (gratuita)
1. Acesse **https://aistudio.google.com/apikey** com uma conta Google e clique em **Create API key**.
2. No Colab, clique no ícone de **chave 🔑 (Secrets)** na barra lateral esquerda → **Add new secret**.
3. Nome: `GEMINI_API_KEY` · Valor: a chave copiada · ative **Notebook access**.


In [11]:
import os
import getpass
import pandas as pd
from typing import List, Annotated, Literal
from typing_extensions import TypedDict
import operator
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import SKLearnVectorStore
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from pydantic import BaseModel, Field
from langchain_community.document_loaders import WebBaseLoader
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import Literal


# Set API keys
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY_CHAT")

# if not GEMINI_API_KEY or not TAVILY_API_KEY:
#     raise EnvironmentError(
#         "Defina GEMINI_API_KEY e TAVILY_API_KEY no arquivo .env."
#     )

print("✅ Variáveis de ambiente carregadas do arquivo .env.")

✅ Variáveis de ambiente carregadas do arquivo .env.


In [6]:
# #@title 2. Ler a chave da API
# try:
#     from google.colab import userdata
#     GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
# except Exception:
#     from getpass import getpass
#     GEMINI_API_KEY = getpass("Cole sua GEMINI_API_KEY: ")
# print("Chave carregada ✅" if GEMINI_API_KEY else "⚠️ Chave não encontrada")

## 3. Base de dados
Baixa a base **Online Retail** (541.909 transações de um varejista on-line britânico, dez/2010 a dez/2011) e faz a **preparação dos dados** (fase do CRISP-DM): remove registros sem cliente, devoluções e valores inválidos. Ler o Excel leva cerca de 1 minuto.

In [7]:
#@title 3. Baixar e preparar a base Online Retail (UCI)
import os, zipfile, urllib.request
import numpy as np
import pandas as pd

URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

def gerar_base_ficticia(n_clientes=1500, seed=42):
    """Base de exemplo com as mesmas colunas da Online Retail (usada só se o download falhar)."""
    rng = np.random.default_rng(seed)
    datas = pd.date_range("2010-12-01", "2011-12-09", freq="h")
    produtos = [(f"P{i:04d}", f"PRODUTO {i}", round(float(rng.uniform(0.5, 15)), 2)) for i in range(300)]
    paises = ["United Kingdom"] * 8 + ["Germany", "France", "EIRE", "Spain", "Netherlands"]
    linhas, nota = [], 536365
    for c in range(n_clientes):
        cid, pais = 12346 + c, rng.choice(paises)
        ativo_ate = rng.uniform(0.3, 1.0)  # alguns clientes param de comprar
        for _ in range(int(rng.geometric(0.15))):
            d = datas[int(rng.uniform(0, ativo_ate) * (len(datas) - 1))]
            nota += 1
            for _ in range(int(rng.integers(1, 12))):
                cod, desc, preco = produtos[int(rng.integers(0, len(produtos)))]
                linhas.append((str(nota), cod, desc, int(rng.integers(1, 24)), d, preco, float(cid), pais))
    return pd.DataFrame(linhas, columns=["InvoiceNo", "StockCode", "Description", "Quantity",
                                         "InvoiceDate", "UnitPrice", "CustomerID", "Country"])

try:
    if not os.path.exists("Online Retail.xlsx"):
        urllib.request.urlretrieve(URL, "online_retail.zip")
        zipfile.ZipFile("online_retail.zip").extractall(".")
    bruto = pd.read_excel("Online Retail.xlsx", dtype={"InvoiceNo": str, "StockCode": str})
    print(f"Base real carregada: {len(bruto):,} linhas")
except Exception as erro:
    print(f"⚠️ Não consegui baixar a base real ({erro}). Usando base fictícia de exemplo.")
    bruto = gerar_base_ficticia()

# Preparação (fase de preparação de dados do CRISP-DM)
df = bruto.dropna(subset=["CustomerID"]).copy()                 # sem cliente identificado
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]        # remove devoluções/cancelamentos
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]            # remove valores inconsistentes
df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Valor"] = df["Quantity"] * df["UnitPrice"]
print(f"Após a limpeza: {len(df):,} linhas | {df['CustomerID'].nunique():,} clientes")
df.head()

Base real carregada: 541,909 linhas
Após a limpeza: 397,884 linhas | 4,338 clientes


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Valor
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## 4. Ferramentas do agente
Cada função abaixo é uma **ferramenta**. O Gemini lê o **nome**, os **tipos dos parâmetros** e a **docstring** (o texto entre `"""`) para entender quando e como usar cada uma. Por isso as docstrings estão bem descritas.

As ferramentas devolvem **apenas resultados agregados** (médias, totais, métricas). Os dados brutos dos clientes não são enviados ao modelo de IA, o que é uma boa prática de privacidade (LGPD).

In [8]:
#@title 4. Ferramentas de mineração de dados que o agente pode usar
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# Guarda os resultados entre uma pergunta e outra (memória do agente)
MEMORIA = {}

def _log(nome, **args):
    print(f"   🔧 Agente usou a ferramenta: {nome}({', '.join(f'{k}={v!r}' for k, v in args.items())})")

def _json(obj):
    return json.dumps(obj, ensure_ascii=False, default=lambda x: x.item() if hasattr(x, "item") else str(x))


def resumo_da_base() -> str:
    """Retorna um resumo geral da base de vendas: período, número de clientes,
    pedidos, produtos, países e faturamento total. Use para perguntas gerais sobre os dados."""
    _log("resumo_da_base")
    return _json({
        "periodo": f"{df['InvoiceDate'].min():%d/%m/%Y} a {df['InvoiceDate'].max():%d/%m/%Y}",
        "linhas": len(df),
        "clientes": df["CustomerID"].nunique(),
        "pedidos": df["InvoiceNo"].nunique(),
        "produtos": df["StockCode"].nunique(),
        "paises": df["Country"].nunique(),
        "faturamento_total": round(df["Valor"].sum(), 2),
        "ticket_medio_por_pedido": round(df.groupby("InvoiceNo")["Valor"].sum().mean(), 2),
    })


def vendas_por_mes() -> str:
    """Retorna o faturamento e o número de pedidos de cada mês. Use para perguntas
    sobre evolução, tendência ou sazonalidade das vendas."""
    _log("vendas_por_mes")
    m = (df.assign(mes=df["InvoiceDate"].dt.to_period("M").astype(str))
           .groupby("mes").agg(faturamento=("Valor", "sum"), pedidos=("InvoiceNo", "nunique")))
    m["faturamento"] = m["faturamento"].round(2)
    return _json(m.reset_index().to_dict(orient="records"))


def ranking(dimensao: str, n: int = 10) -> str:
    """Retorna os n maiores itens por faturamento em uma dimensão.

    Args:
        dimensao: uma destas opções: 'produto', 'pais' ou 'cliente'.
        n: quantos itens mostrar (padrão 10).
    """
    _log("ranking", dimensao=dimensao, n=n)
    col = {"produto": "Description", "pais": "Country", "cliente": "CustomerID"}.get(dimensao)
    if col is None:
        return _json({"erro": "dimensao deve ser 'produto', 'pais' ou 'cliente'"})
    r = df.groupby(col)["Valor"].sum().sort_values(ascending=False).head(int(n)).round(2)
    return _json([{dimensao: str(k), "faturamento": v} for k, v in r.items()])


def _rfm(dados, data_ref):
    rfm = dados.groupby("CustomerID").agg(
        recencia=("InvoiceDate", lambda s: (data_ref - s.max()).days),
        frequencia=("InvoiceNo", "nunique"),
        monetario=("Valor", "sum"),
    )
    return rfm[rfm["monetario"] > 0]


def _features(dados, data_ref):
    X = _rfm(dados, data_ref)
    X["ticket_medio"] = X["monetario"] / X["frequencia"]
    X["produtos_distintos"] = dados.groupby("CustomerID")["StockCode"].nunique()
    X["dias_de_cliente"] = dados.groupby("CustomerID")["InvoiceDate"].agg(lambda s: (data_ref - s.min()).days)
    return X


def segmentar_clientes(k: int = 0) -> str:
    """Segmenta os clientes com o algoritmo k-means aplicado às variáveis RFM
    (recência, frequência e valor monetário). Retorna o perfil médio de cada grupo
    e o coeficiente de silhueta. Use para perguntas sobre perfis, grupos ou segmentos de clientes.

    Args:
        k: número de grupos. Use 0 para o agente escolher automaticamente o melhor k entre 2 e 8 pela silhueta.
    """
    _log("segmentar_clientes", k=k)
    data_ref = df["InvoiceDate"].max() + pd.Timedelta(days=1)
    rfm = _rfm(df, data_ref)
    # log reduz a assimetria de frequência e valor; depois padroniza (z-score)
    X = StandardScaler().fit_transform(np.log1p(rfm[["recencia", "frequencia", "monetario"]]))
    amostra = np.random.RandomState(42).choice(len(X), min(5000, len(X)), replace=False)

    silhuetas = {}
    for kk in ([int(k)] if k and int(k) >= 2 else range(2, 9)):
        rotulos = KMeans(n_clusters=kk, n_init=10, random_state=42).fit_predict(X)
        silhuetas[kk] = round(silhouette_score(X[amostra], rotulos[amostra]), 3)
    melhor_k = max(silhuetas, key=silhuetas.get)

    rfm["segmento"] = KMeans(n_clusters=melhor_k, n_init=10, random_state=42).fit_predict(X)
    perfil = rfm.groupby("segmento").agg(
        clientes=("recencia", "size"), recencia_media_dias=("recencia", "mean"),
        frequencia_media=("frequencia", "mean"), valor_medio=("monetario", "mean"),
        valor_total=("monetario", "sum")).round(1)
    MEMORIA["rfm"] = rfm
    return _json({"k_escolhido": melhor_k, "silhueta_por_k": silhuetas,
                  "perfil_dos_segmentos": perfil.reset_index().to_dict(orient="records"),
                  "dica": "Dê nomes de negócio aos segmentos (ex.: campeões, em risco, novos) com base no perfil."})


def prever_churn(meses_observacao: int = 3, profundidade_maxima: int = 4) -> str:
    """Treina uma árvore de decisão para prever churn: clientes que compraram antes,
    mas NÃO compraram nos últimos meses da base. Retorna métricas de avaliação no
    conjunto de teste, matriz de confusão, importância das variáveis e as regras da árvore.

    Args:
        meses_observacao: tamanho, em meses, da janela final usada para verificar se o cliente voltou a comprar (padrão 3).
        profundidade_maxima: profundidade máxima da árvore; valores menores evitam overfitting (padrão 4).
    """
    _log("prever_churn", meses_observacao=meses_observacao, profundidade_maxima=profundidade_maxima)
    corte = df["InvoiceDate"].max() - pd.DateOffset(months=int(meses_observacao))
    antes, depois = df[df["InvoiceDate"] <= corte], df[df["InvoiceDate"] > corte]
    X = _features(antes, corte)
    y = (~X.index.isin(depois["CustomerID"].unique())).astype(int)  # 1 = churn

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    arvore = DecisionTreeClassifier(max_depth=int(profundidade_maxima), class_weight="balanced",
                                    min_samples_leaf=20, random_state=42).fit(Xtr, ytr)
    p = arvore.predict(Xte)
    # aplica o modelo aos dados ATUAIS (base inteira) para estimar o risco daqui para frente
    atual = _features(df, df["InvoiceDate"].max() + pd.Timedelta(days=1))
    atual["prob_churn"] = arvore.predict_proba(atual[X.columns])[:, 1]
    MEMORIA["churn"] = atual

    from sklearn.tree import export_text
    return _json({
        "clientes_analisados": len(X), "taxa_de_churn": round(y.mean(), 3),
        "acuracia": round(accuracy_score(yte, p), 3), "precisao": round(precision_score(yte, p), 3),
        "revocacao_recall": round(recall_score(yte, p), 3), "f1": round(f1_score(yte, p), 3),
        "matriz_confusao": {"linhas=real [fica, churn]": confusion_matrix(yte, p).tolist()},
        "importancia_variaveis": dict(sorted(zip(X.columns, arvore.feature_importances_.round(3)),
                                             key=lambda t: -t[1])),
        "regras_da_arvore": export_text(arvore, feature_names=list(X.columns), decimals=1)[:2500],
    })


def clientes_em_risco(n: int = 10) -> str:
    """Lista os n clientes ATUAIS com maior probabilidade de churn (deixar de comprar) segundo a árvore de decisão.
    Se o modelo ainda não foi treinado, treina com os parâmetros padrão.

    Args:
        n: quantos clientes listar (padrão 10).
    """
    _log("clientes_em_risco", n=n)
    if "churn" not in MEMORIA:
        prever_churn()
    t = MEMORIA["churn"].sort_values(["prob_churn", "monetario"], ascending=False).head(int(n))
    return _json(t.reset_index()[["CustomerID", "prob_churn", "recencia", "frequencia", "monetario"]]
                 .round(2).to_dict(orient="records"))


def exportar_para_power_bi() -> str:
    """Gera o arquivo clientes_para_power_bi.csv com segmento e probabilidade de churn
    de cada cliente, pronto para importar no Power BI. Roda a segmentação e o modelo de churn se ainda não tiverem sido executados."""
    _log("exportar_para_power_bi")
    if "rfm" not in MEMORIA:
        segmentar_clientes()
    if "churn" not in MEMORIA:
        prever_churn()
    saida = MEMORIA["rfm"].join(MEMORIA["churn"][["prob_churn"]].round(3), how="left").reset_index()
    saida.to_csv("clientes_para_power_bi.csv", index=False, sep=";", decimal=",")
    return _json({"arquivo": "clientes_para_power_bi.csv", "linhas": len(saida),
                  "colunas": list(saida.columns)})


FERRAMENTAS = [resumo_da_base, vendas_por_mes, ranking, segmentar_clientes,
               prever_churn, clientes_em_risco, exportar_para_power_bi]
print(f"✅ {len(FERRAMENTAS)} ferramentas prontas:", [f.__name__ for f in FERRAMENTAS])

✅ 7 ferramentas prontas: ['resumo_da_base', 'vendas_por_mes', 'ranking', 'segmentar_clientes', 'prever_churn', 'clientes_em_risco', 'exportar_para_power_bi']


## 5. Criar o agente
O `chat` guarda o histórico da conversa, então dá pra fazer perguntas de acompanhamento (“e o segmento 2?”). A chamada das ferramentas é **automática**: o SDK executa a função que o Gemini pediu, devolve o resultado e o modelo continua até ter a resposta final.

In [9]:
#@title 5. Criar o agente
from google import genai
from google.genai import types
from IPython.display import Markdown, display

# Modelo com plano gratuito. Se der erro de "model not found", troque por outro da lista:
# https://ai.google.dev/gemini-api/docs/models
MODELO = "gemini-3.5-flash"

INSTRUCOES = """Você é um analista de Business Intelligence especializado em mineração de dados,
trabalhando com a base de vendas de um varejista on-line britânico (valores em libras, £).

Regras:
- SEMPRE use as ferramentas para obter números. Nunca invente valores.
- Responda em português do Brasil, de forma clara e objetiva, como para um gestor.
- Ao apresentar segmentos de clientes, dê a cada um um nome de negócio (ex.: "Campeões",
  "Em risco", "Ocasionais") com base em recência, frequência e valor.
- Ao falar do modelo de churn, explique precisão, recall e F1 em linguagem simples e diga
  se o modelo é confiável. Lembre que acurácia sozinha engana quando as classes são desbalanceadas.
- Termine com 2 ou 3 ações práticas recomendadas.
"""

client = genai.Client(api_key=GEMINI_API_KEY)
config = types.GenerateContentConfig(
    system_instruction=INSTRUCOES,
    tools=FERRAMENTAS,          # as funções Python da etapa 4
    temperature=0.2,            # respostas mais estáveis
    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=10),
)

def novo_chat():
    """Começa uma conversa do zero (apaga o histórico)."""
    global chat
    chat = client.chats.create(model=MODELO, config=config)

def perguntar(pergunta):
    print(f"🧑 {pergunta}\n")
    resposta = chat.send_message(pergunta)
    print()
    display(Markdown(resposta.text or "_(o agente não retornou texto)_"))

novo_chat()
print(f"Agente pronto ✅  (modelo: {MODELO})")

Agente pronto ✅  (modelo: gemini-3.5-flash)


## 6. Conversando com o agente
As linhas 🔧 mostram quais ferramentas o agente decidiu usar. Troquem as perguntas à vontade.

In [12]:
perguntar("Me dá um panorama geral da base de vendas e de como o faturamento evoluiu mês a mês.")

🧑 Me dá um panorama geral da base de vendas e de como o faturamento evoluiu mês a mês.




Aqui está um panorama geral dos dados de vendas do varejista on-line, cobrindo o período de **01/12/2010 a 09/12/2011**.

---

### 📊 1. Panorama Geral da Base

* **Faturamento Total:** £ 8.911.407,90
* **Total de Pedidos:** 22.190
* **Base de Clientes Únicos:** 4.372
* **Variedade de Produtos (SKUs):** 3.684
* **Alcance Geográfico:** 37 países

---

### 📈 2. Evolução Mensal do Faturamento e Volume de Pedidos

| Mês/Ano | Faturamento (£) | Nº de Pedidos | Destaques |
| :--- | :---: | :---: | :--- |
| **12/2010** | £ 748.957,02 | 2.020 | Período festivo inicial |
| **01/2011** | £ 560.000,22 | 1.261 | Desaceleração pós-Festas |
| **02/2011** | £ 498.062,65 | 1.235 | Menor faturamento do ano |
| **03/2011** | £ 683.223,44 | 1.718 | Recuperação no final do Q1 |
| **04/2011** | £ 493.207,12 | 1.396 | Estabilidade de baixa |
| **05/2011** | £ 723.333,51 | 1.844 | Início do aquecimento no Q2 |
| **06/2011** | £ 691.122,38 | 1.707 | Média estável |
| **07/2011** | £ 681.300,11 | 1.751 | Média estável |
| **08/2011** | £ 682.680,51 | 1.737 | Preparações antecipadas |
| **09/2011** | £ 1.019.687,62 | 2.078 | Primeiro mês a ultrapassar £ 1 milhão |
| **10/2011** | £ 1.070.715,08 | 2.262 | Crescimento contínuo no Q4 |
| **11/2011** | £ 1.461.756,25 | 3.046 | **Pico de vendas do ano** (Black Friday/Festas) |
| **12/2011\*** | £ 433.861,99 | 935 | *Dados parciais até 09/12/2011* |

#### **Principais Observações da Evolução:**
1. **Forte Sazonalidade de Fim de Ano (Q4):** O faturamento dispara a partir de setembro, culminando em novembro com o recorde de **£ 1,46 milhão** (quase o dobro da média dos meses anteriores).
2. **Período de Baixa no Início do Ano (Q1):** Fevereiro e abril registraram os menores faturamentos do período (~£ 490k-£ 498k).
3. **Platô de Estabilidade (Maio a Agosto):** As vendas mantiveram-se regulares na faixa de £ 680k a £ 720k.

---

### 💡 Recomendações Práticas

1. **Planejamento Antecipado de Estoque e Logística para o Q4:** Como os meses de setembro a novembro respondem pela maior fatia do faturamento, o planejamento de compras e capacidade operacional deve ser ajustado com 3 a 4 meses de antecedência.
2. **Campanhas de Estímulo para os Meses de Baixa (Fevereiro/Abril):** Desenvolver promoções temáticas ou ofertas de queima de estoque no início do ano para mitigar a queda sazonal do primeiro trimestre.
3. **Estratégia de Retenção Pós-Pico:** Criar fluxos automatizados de e-mail marketing em janeiro para reengajar os novos clientes captados durante o pico de vendas de final de ano.

In [14]:
perguntar("Quais perfis de clientes existem? Faça a segmentação e dê um nome para cada grupo.")

🧑 Quais perfis de clientes existem? Faça a segmentação e dê um nome para cada grupo.

   🔧 Agente usou a ferramenta: segmentar_clientes(k=0)
   🔧 Agente usou a ferramenta: segmentar_clientes(k=3)


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
perguntar("Quais clientes têm mais risco de parar de comprar? O modelo é confiável? O que mais influencia o churn?")

🧑 Quais clientes têm mais risco de parar de comprar? O modelo é confiável? O que mais influencia o churn?

   🔧 Agente usou a ferramenta: prever_churn(meses_observacao=3, profundidade_maxima=4)
   🔧 Agente usou a ferramenta: clientes_em_risco(n=10)



Com base no treinamento de um modelo de inteligência artificial (Árvore de Decisão) para prever o **churn** (clientes que correm o risco de parar de comprar), apresento a análise de confiabilidade, os fatores de influência e a lista dos clientes mais críticos.

---

### 1. O modelo é confiável? (Análise das Métricas)
Para avaliar o modelo, não olhamos apenas para a acurácia geral (que é de **63,8%**), pois ela pode ser enganosa. Analisamos três métricas fundamentais sob a ótica do seu negócio:

*   **Revocação / Recall (74,9%):** Esta é a métrica mais importante para nós. Ela mostra que **o modelo consegue identificar 75% de todos os clientes que realmente vão dar churn**. Ou seja, deixamos passar pouquíssimos clientes em risco sem aviso.
*   **Precisão (55,8%):** Significa que de cada 100 clientes que o modelo aponta como "em risco", cerca de 56 realmente iriam embora (e 44 continuariam comprando). Para o marketing, esse "erro" (falso positivo) é perfeitamente aceitável, pois o custo de enviar uma oferta preventiva para quem já ia comprar é muito menor do que o custo de perder um cliente real.
*   **F1-Score (64,0%):** É o equilíbrio entre a precisão e o recall. 

**Veredito:** **Sim, o modelo é altamente confiável e estratégico.** Ele funciona como uma excelente "rede de segurança", garantindo que a grande maioria dos clientes prestes a abandonar a marca seja detectada a tempo para uma ação preventiva.

---

### 2. O que mais influencia o Churn?
O modelo revelou que o comportamento de compra é muito mais decisivo para o abandono do que o valor gasto em si.

1.  **Frequência de Compra (77,2% de importância):** É, de longe, o fator mais crítico. Clientes que compram poucas vezes têm uma tendência esmagadora de não voltar.
2.  **Variedade de Produtos (9,9% de importância):** Clientes que compram poucos produtos distintos (menos de 11 itens diferentes) tendem a abandonar a marca com muito mais facilidade.
3.  **Ticket Médio (4,8%) e Valor Total Gasto (4,6%):** O valor financeiro tem pouca influência direta no churn se comparado à frequência.

**A regra de ouro da árvore:** Se um cliente tem uma frequência menor ou igual a 2 compras e comprou menos de 11 produtos diferentes, o risco de churn é praticamente certo.

---

### 3. Clientes Atuais com Maior Risco de Churn
Abaixo estão os 10 clientes com maior probabilidade de abandono (**91% de risco**). Note que alguns deles gastaram valores altíssimos, o que torna a perda ainda mais dolorosa:

| ID do Cliente | Probabilidade de Churn | Recência (Dias sem comprar) | Frequência (Pedidos) | Valor Total Gasto (£) |
| :---: | :---: | :---: | :---: | :---: |
| **16446** | **91%** | 1 | 2 | £ 168.472,50 |
| **12346** | **91%** | 326 | 1 | £ 77.183,60 |
| **18087** | **91%** | 98 | 2 | £ 4.055,72 |
| **12755** | **91%** | 281 | 2 | £ 3.811,95 |
| **13135** | **91%** | 197 | 1 | £ 3.096,00 |
| **16754** | **91%** | 372 | 1 | £ 2.002,40 |
| **16698** | **91%** | 227 | 2 | £ 1.998,00 |
| **14935** | **91%** | 297 | 1 | £ 1.784,71 |
| **17353** | **91%** | 129 | 2 | £ 1.740,00 |
| **13791** | **91%** | 128 | 2 | £ 1.516,00 |

---

### 🎯 Ações Práticas Recomendadas

1.  **Atenção Imediata ao Cliente 16446:** Este cliente realizou uma compra gigantesca de **£ 168.472,50** ontem (recência de 1 dia), mas o modelo o acusa com alto risco de churn porque ele tem baixíssima frequência (apenas 2 pedidos). *Ação:* A equipe de contas ou comercial deve entrar em contato imediatamente para agradecer a compra, garantir que a entrega seja perfeita e tentar fechar um contrato de fornecimento recorrente.
2.  **Foco na "Segunda Compra" (Onboarding de Clientes):** Como a frequência dita 77% do churn, a sua maior prioridade de marketing deve ser fazer o cliente de primeira viagem comprar pela segunda vez. Crie uma régua de e-mails de boas-vindas com um cupom de desconto agressivo para o segundo pedido, válido por apenas 30 dias.
3.  **Estímulo à Diversificação de Produtos (Cross-Selling):** Clientes que compram poucos tipos de produtos diferentes dão mais churn. Incentive a navegação por outras categorias do site. Se ele comprou um produto da categoria A, envie recomendações personalizadas das categorias B e C.

In [ ]:
perguntar("Gere o arquivo para eu montar o dashboard no Power BI.")

🧑 Gere o arquivo para eu montar o dashboard no Power BI.

   🔧 Agente usou a ferramenta: exportar_para_power_bi()



O arquivo **`clientes_para_power_bi.csv`** foi gerado com sucesso e está pronto para ser importado no seu Power BI!

Este arquivo consolida as informações mais valiosas de cada um dos **4.338 clientes** da sua base, unindo os dados transacionais com a inteligência analítica que geramos anteriormente.

---

### 📋 Estrutura do Arquivo Exportado
O arquivo contém as seguintes colunas:
1.  **`CustomerID`:** Identificador único do cliente (chave de ligação).
2.  **`recencia`:** Quantidade de dias desde a última compra.
3.  **`frequencia`:** Quantidade total de pedidos realizados.
4.  **`monetario`:** Valor total gasto pelo cliente (£).
5.  **`segmento`:** Grupo do cliente (0 para "Adormecidos/Ocasionais" e 1 para "Campeões Recorrentes/VIPs").
6.  **`prob_churn`:** Probabilidade calculada pelo modelo de o cliente parar de comprar (de 0.00 a 1.00).

---

### 💡 Dicas para Montar seu Dashboard no Power BI

Para criar um painel de alto impacto para a diretoria, sugiro estruturar três visões principais:

1.  **Matriz de Risco (Cruzamento de Segmento vs. Churn):**
    *   Crie um gráfico de dispersão (*Scatter Chart*) colocando a **Frequência** no eixo X, o **Valor Monetário** no eixo Y e a cor dos pontos baseada na **Probabilidade de Churn**.
    *   Isso permitirá identificar visualmente os clientes "VIPs" (alto valor) que estão na zona de perigo (cor vermelha/alta probabilidade de churn).

2.  **Painel de Alerta de Churn (Foco em CS/Vendas):**
    *   Crie uma tabela com filtros rápidos onde a equipe comercial possa filtrar clientes com **`prob_churn` > 80%** e **`monetario` > £ 1.000**.
    *   Esta será a lista de "ligação imediata" para o time de Customer Success ou comercial salvar essas contas.

3.  **KPIs de Saúde da Base:**
    *   **Cartão 1:** Faturamento sob Risco (soma do valor monetário de clientes com probabilidade de churn > 70%).
    *   **Cartão 2:** Quantidade de Clientes VIPs Ativos (Segmento 1 com baixa probabilidade de churn).

---

### 🎯 Ações Práticas Recomendadas

1.  **Relacionamento de Tabelas no Power BI:** Ao importar o arquivo, crie um relacionamento de **1 para Muitos (1:*)** entre a coluna `CustomerID` deste arquivo CSV e a sua tabela de vendas/itens original. Isso permitirá filtrar todo o histórico de compras de um cliente apenas clicando no nome dele no painel de risco.
2.  **Atualização Periódica:** Agende a execução deste modelo mensalmente para atualizar as probabilidades de churn e os segmentos, garantindo que o painel do Power BI reflita sempre o comportamento mais recente dos clientes.

### Baixar o arquivo para o Power BI
No Power BI Desktop: **Obter dados → Texto/CSV**. O arquivo usa `;` como separador e `,` como decimal (padrão brasileiro).

In [ ]:
from google.colab import files
files.download("clientes_para_power_bi.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
MODELO = "gemini-3.1-flash-lite"

In [ ]:
print("Modelos disponíveis para uso:")
try:
    for model in client.models.list():
        print(f"- {model.name}: {model.display_name}")
except Exception as e:
    print(f"Erro ao listar modelos: {e}")

Modelos disponíveis para uso:
- models/gemini-2.5-flash: Gemini 2.5 Flash
- models/gemini-2.5-pro: Gemini 2.5 Pro
- models/gemini-2.5-flash-preview-tts: Gemini 2.5 Flash Preview TTS
- models/gemini-2.5-pro-preview-tts: Gemini 2.5 Pro Preview TTS
- models/gemma-4-26b-a4b-it: Gemma 4 26B A4B IT
- models/gemma-4-31b-it: Gemma 4 31B IT
- models/gemini-flash-latest: Gemini Flash Latest
- models/gemini-flash-lite-latest: Gemini Flash-Lite Latest
- models/gemini-pro-latest: Gemini Pro Latest
- models/gemini-2.5-flash-lite: Gemini 2.5 Flash-Lite
- models/gemini-2.5-flash-image: Nano Banana
- models/gemini-3-flash-preview: Gemini 3 Flash Preview
- models/gemini-3.1-pro-preview: Gemini 3.1 Pro Preview
- models/gemini-3.1-pro-preview-customtools: Gemini 3.1 Pro Preview Custom Tools
- models/gemini-3.1-flash-lite-preview: Gemini 3.1 Flash Lite Preview
- models/gemini-3.1-flash-lite: Gemini 3.1 Flash Lite
- models/gemini-3-pro-image-preview: Nano Banana Pro
- models/gemini-3-pro-image: Nano Banana 

## 7. Bônus: dois agentes (analista + revisor)
Um padrão comum é colocar um **segundo agente** para revisar o trabalho do primeiro. Aqui o **revisor** não tem acesso às ferramentas: ele só lê a resposta do analista e aponta problemas, como conclusões exageradas, números sem fonte ou limitações não mencionadas.

In [ ]:
#@title 7. Analista + revisor
REVISOR = types.GenerateContentConfig(
    system_instruction="""Você é um professor revisor de trabalhos de mineração de dados.
Leia a pergunta e a resposta de um analista e aponte, em no máximo 5 tópicos curtos:
afirmações sem evidência, conclusões exageradas, problemas metodológicos (ex.: acurácia em
classes desbalanceadas, dados de um só ano/varejista) e o que poderia ser melhorado. Responda em português.""",
    temperature=0.2,
)

def analise_com_revisao(pergunta):
    analista = client.chats.create(model=MODELO, config=config)
    print(f"🧑 {pergunta}\n")
    r1 = analista.send_message(pergunta).text
    display(Markdown("### 📊 Analista\n" + r1))
    r2 = client.models.generate_content(
        model=MODELO, config=REVISOR,
        contents=f"Pergunta: {pergunta}\n\nResposta do analista:\n{r1}",
    ).text
    display(Markdown("### 🔍 Revisor\n" + r2))

analise_com_revisao("Vale a pena fazer uma campanha de retenção? Para quais clientes?")

🧑 Vale a pena fazer uma campanha de retenção? Para quais clientes?

   🔧 Agente usou a ferramenta: resumo_da_base()
   🔧 Agente usou a ferramenta: segmentar_clientes(k=0)
   🔧 Agente usou a ferramenta: prever_churn(meses_observacao=3, profundidade_maxima=4)


### 📊 Analista
Sim, uma campanha de retenção é altamente recomendada. Analisando a base de dados, identificamos dois perfis distintos de clientes e um risco de churn (perda de clientes) que impacta significativamente o faturamento.

### Diagnóstico dos Clientes
Dividimos sua base em dois grupos principais:

1.  **"Campeões" (1.667 clientes):** São clientes fiéis, com alta frequência (média de 8,4 compras) e recência recente (média de 26 dias). Eles representam a maior parte do seu faturamento (£ 7,5 milhões).
2.  **"Em Risco / Ocasionais" (2.671 clientes):** Compram pouco (média de 1,7 vezes) e não compram há muito tempo (média de 134 dias). Este grupo é o foco principal para ações de reativação.

### Sobre o Modelo de Churn
Treinamos um modelo para prever quais clientes estão prestes a parar de comprar.
*   **Confiabilidade:** O modelo apresenta um **Recall de 74,9%**, o que significa que ele consegue identificar corretamente cerca de 3 em cada 4 clientes que realmente entrariam em churn. O **F1-Score de 64%** indica um equilíbrio aceitável entre precisão e abrangência. É uma ferramenta confiável para priorizar esforços.
*   **O que mais importa:** A variável que mais define se um cliente vai embora é a **frequência**. Clientes que compraram poucas vezes são os mais propensos a abandonar a marca.

### Recomendações Práticas

1.  **Campanha de Reativação para o segmento "Em Risco":** Não tente recuperar todos de uma vez. Foque naqueles que o modelo de churn identificou com alta probabilidade de saída, mas que possuem um histórico de compra de produtos variados (o que indica potencial de valor). Ofereça um incentivo para a "segunda compra", já que a baixa frequência é o principal gatilho de churn.
2.  **Programa de Fidelidade para os "Campeões":** Como este grupo sustenta o negócio, o objetivo aqui não é retenção por risco, mas **proteção e aumento de ticket**. Crie um programa de recompensas ou acesso antecipado a produtos para garantir que eles não migrem para a concorrência.
3.  **Monitoramento de Frequência:** Como a frequência é o fator determinante para o churn, implemente um gatilho automático: se um cliente que comprou apenas uma ou duas vezes passar de 60 dias sem nova interação, dispare uma comunicação personalizada (e-mail ou oferta) antes que ele entre na zona de "risco crítico".

### 🔍 Revisor
Como professor revisor, aqui estão os pontos críticos sobre a sua análise:

1.  **Viés de causalidade:** A afirmação de que a "frequência" é o fator que *define* o churn confunde correlação com causalidade; clientes que compram pouco podem estar em churn justamente por não terem gostado da primeira experiência, e não o contrário.
2.  **Problema metodológico (Métricas):** O uso de F1-Score de 64% é preocupante. Em modelos de churn, que geralmente lidam com classes desbalanceadas, você deve apresentar a **Matriz de Confusão** ou a **Curva Precision-Recall** para garantir que o modelo não está apenas "chutando" a classe majoritária.
3.  **Ausência de evidência de ROI:** A recomendação de campanha é genérica. Falta uma análise de custo-benefício: o custo de aquisição/retenção desses clientes "Em Risco" é menor do que o valor esperado (LTV) que eles trarão?
4.  **Limitação de dados:** Não há menção sobre a sazonalidade ou o período da base. Se os dados forem de um único ano ou período promocional, o comportamento observado pode não ser replicável em outros contextos.
5.  **Sugestão de melhoria:** Realize um teste A/B antes de escalar a campanha de reativação e inclua uma análise de *Feature Importance* mais robusta (como SHAP values) para validar se a frequência é realmente o preditor mais forte ou se há variáveis de comportamento de navegação/engajamento omitidas.

In [ ]:
#@title 8. Ferramenta: criar dashboard interativo
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CORES_SEG = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CORES_RISCO = {"Alto": "#d03b3b", "Médio": "#ec835a", "Baixo": "#0ca30c", "Sem previsão": "#898781"}
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

def _nomear_segmentos(base):
    """Dá nomes de negócio aos segmentos ordenando por RFM (menor recência, maior frequência e valor = melhor)."""
    perfil = base.groupby("segmento")[["recencia", "frequencia", "monetario"]].median()
    score = -perfil["recencia"].rank() + perfil["frequencia"].rank() + perfil["monetario"].rank()
    ordem = score.sort_values(ascending=False).index.tolist()
    nomes = {2: ["Alto valor", "Baixo valor"],
             3: ["Campeões", "Regulares", "Inativos"],
             4: ["Campeões", "Fiéis", "Em esfriamento", "Inativos"]}.get(
        len(ordem),
        ["Campeões", "Fiéis", "Promissores", "Regulares", "Em esfriamento", "Em risco", "Hibernando"][:len(ordem) - 1] + ["Inativos"])
    return {seg: nome for seg, nome in zip(ordem, nomes)}


def criar_dashboard(segmento: str = "todos") -> str:
    """Monta e exibe na tela um dashboard interativo de clientes com indicadores (KPIs),
    faturamento mensal, receita por segmento, mapa RFM, risco de churn por segmento e a
    lista de clientes prioritários para retenção. Roda a segmentação e o modelo de churn
    se ainda não tiverem sido executados. Use quando pedirem dashboard, painel ou gráficos.

    Args:
        segmento: 'todos' para a base inteira, ou o nome do segmento para filtrar (ex.: 'Campeões', 'Inativos').
    """
    _log("criar_dashboard", segmento=segmento)
    if "rfm" not in MEMORIA:
        segmentar_clientes()
    if "churn" not in MEMORIA:
        prever_churn()

    # Junta segmento + probabilidade de churn (os mesmos dados do CSV do Power BI)
    base = MEMORIA["rfm"].join(MEMORIA["churn"][["prob_churn"]], how="left").reset_index()
    nomes = _nomear_segmentos(base)
    base["nome_segmento"] = base["segmento"].map(nomes)
    base["risco"] = (pd.cut(base["prob_churn"], [-0.01, 0.4, 0.7, 1.0], labels=["Baixo", "Médio", "Alto"])
                       .astype(object).where(base["prob_churn"].notna(), "Sem previsão"))
    cor_seg = {nomes[s]: CORES_SEG[i % 8] for i, s in enumerate(sorted(nomes))}  # cor fixa por segmento

    # Filtro opcional por segmento
    titulo = "Todos os clientes"
    if str(segmento).strip().lower() not in ("todos", "", "all"):
        alvo = str(segmento).strip().lower()
        filtro = base["nome_segmento"].str.lower().str.contains(alvo) | base["segmento"].astype(str).eq(alvo)
        if not filtro.any():
            return _json({"erro": f"Segmento '{segmento}' não encontrado.", "segmentos_disponiveis": list(nomes.values())})
        base = base[filtro]
        titulo = "Segmento: " + ", ".join(base["nome_segmento"].unique())

    n = len(base)
    receita = base["monetario"].sum()
    alto = base[base["risco"] == "Alto"]
    fonte = dict(family="system-ui, Segoe UI, sans-serif", color="#0b0b0b")

     # ---------- 1. KPIs ----------
    kpis = [("Clientes", n, "", ",.0f"), ("Receita total", receita, "£ ", ",.0f"),
            ("Valor médio por cliente", receita / n, "£ ", ",.0f"),
            ("Clientes em alto risco", len(alto) / n * 100, "", ".1f"),
            ("Receita em risco", alto["monetario"].sum(), "£ ", ",.0f")]
    fig_kpi = go.Figure()
    for i, (nome, valor, prefixo, formato) in enumerate(kpis):
        fig_kpi.add_trace(go.Indicator(
            mode="number", value=valor, domain={"row": 0, "column": i},
            title={"text": nome, "font": {"size": 13, "color": "#52514e"}},
            number={"prefix": prefixo, "suffix": " %" if "risco" in nome and "Receita" not in nome else "",
                    "valueformat": formato, "font": {"size": 30}}))
    fig_kpi.update_layout(grid={"rows": 1, "columns": 5}, height=160, separators=",.", font=fonte,
                          title=dict(text=f"<b>Painel de Clientes</b> · {titulo}", x=0, font=dict(size=20)),
                          margin=dict(t=60, b=10, l=20, r=20), paper_bgcolor="#fcfcfb")

    # ---------- 2. Gráficos ----------
    fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.14, vertical_spacing=0.17,
                        subplot_titles=("Faturamento mensal (£)", "Receita por segmento (£)",
                                        "Mapa RFM: recência × frequência", "Risco de churn por segmento (% dos clientes)"))

    # 2a. Faturamento mensal dos clientes filtrados
    vendas = df[df["CustomerID"].isin(base["CustomerID"])]
    mensal = vendas.groupby(vendas["InvoiceDate"].dt.to_period("M"))["Valor"].sum()
    rotulos = [f"{MESES[p.month - 1]}/{str(p.year)[2:]}" for p in mensal.index]
    if vendas["InvoiceDate"].max().day < 28:
        rotulos[-1] += " (parcial)"
    fig.add_trace(go.Scatter(x=rotulos, y=mensal.values, mode="lines+markers", showlegend=False,
                             line=dict(color=CORES_SEG[0], width=2), marker=dict(size=7),
                             hovertemplate="%{x}<br><b>£ %{y:,.0f}</b><extra></extra>"), row=1, col=1)

    # 2b. Receita por segmento
    seg = (base.groupby("nome_segmento").agg(clientes=("CustomerID", "size"), receita=("monetario", "sum"))
               .sort_values("receita").reset_index())
    fig.add_trace(go.Bar(x=seg["receita"], y=seg["nome_segmento"], orientation="h", showlegend=False,
                         marker=dict(color=[cor_seg[s] for s in seg["nome_segmento"]]),
                         customdata=seg["clientes"], texttemplate="£ %{x:,.0f}", textposition="auto",
                         hovertemplate="%{y}<br><b>£ %{x:,.0f}</b><br>%{customdata} clientes<extra></extra>"),
                  row=1, col=2)

    # 2c. Mapa RFM colorido pelo risco de churn
    for faixa in ["Baixo", "Médio", "Alto", "Sem previsão"]:
        d = base[base["risco"] == faixa]
        if d.empty:
            continue
        fig.add_trace(go.Scattergl(
            x=d["recencia"], y=d["frequencia"], mode="markers", name=f"Risco {faixa.lower()}", legendgroup=faixa,
            marker=dict(color=CORES_RISCO[faixa], size=7, opacity=0.65, line=dict(width=0.5, color="white")),
            customdata=np.stack([d["CustomerID"], d["nome_segmento"], d["monetario"],
                                 (d["prob_churn"] * 100).fillna(-1)], axis=-1),
            hovertemplate=("Cliente %{customdata[0]} · %{customdata[1]}<br>Recência: %{x} dias · "
                           "Frequência: %{y} pedidos<br>Valor: £ %{customdata[2]:,.0f}<br>"
                           "<b>Prob. churn: %{customdata[3]:.0f}%</b><extra></extra>")), row=2, col=1)

    # 2d. Distribuição de risco dentro de cada segmento (barras 100% empilhadas)
    tab = base.groupby(["nome_segmento", "risco"]).size().unstack(fill_value=0)
    tab = (tab.div(tab.sum(axis=1), axis=0) * 100).loc[seg["nome_segmento"]]
    for faixa in ["Alto", "Médio", "Baixo", "Sem previsão"]:
        if faixa in tab:
            fig.add_trace(go.Bar(y=tab.index, x=tab[faixa], orientation="h", legendgroup=faixa, showlegend=False,
                                 marker=dict(color=CORES_RISCO[faixa], line=dict(color="#fcfcfb", width=2)),
                                 hovertemplate="%{y}<br>Risco " + faixa.lower() + ": <b>%{x:.0f}%</b><extra></extra>"),
                          row=2, col=2)

    fig.update_layout(height=820, barmode="stack", template="plotly_white", separators=",.", font=fonte,
                      paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", hovermode="closest",
                      legend=dict(orientation="h", x=0, y=-0.1, title_text="Clique para mostrar/ocultar: "),
                      margin=dict(t=50, b=70, l=60, r=30))
    fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    fig.update_yaxes(tickprefix="£ ", row=1, col=1)
    fig.update_xaxes(title_text="Recência (dias desde a última compra)", row=2, col=1)
    fig.update_yaxes(title_text="Frequência (pedidos, escala log)", type="log", row=2, col=1)
    fig.update_xaxes(range=[0, 100], ticksuffix="%", row=2, col=2)

    # ---------- 3. Tabela: clientes prioritários para retenção ----------
    top = alto.sort_values("monetario", ascending=False).head(10)
    fig_tab = go.Figure(go.Table(
        columnwidth=[1, 1.6, 1, 1, 1, 1.2],
        header=dict(values=["Cliente", "Segmento", "Prob. churn", "Recência (dias)", "Pedidos", "Valor (£)"],
                    fill_color="#f0efec", font=dict(color="#0b0b0b", size=12), align="left"),
        cells=dict(values=[top["CustomerID"], top["nome_segmento"], (top["prob_churn"] * 100).round(0).astype(int).astype(str) + "%",
                           top["recencia"], top["frequencia"], top["monetario"].map(lambda v: f"{v:,.0f}".replace(",", "."))],
                   fill_color="#fcfcfb", line_color="#e1e0d9", align="left", height=26)))
    fig_tab.update_layout(title=dict(text="Clientes prioritários para retenção (alto risco, maior valor)", x=0, font=dict(size=15)),
                          height=80 + 28 * max(len(top), 1) + 40, margin=dict(t=40, b=10, l=10, r=10),
                          font=fonte, paper_bgcolor="#fcfcfb")

    fig_kpi.show(); fig.show(); fig_tab.show()

    # Resumo que volta para o agente comentar
    resumo_seg = (base.groupby("nome_segmento")
                      .agg(clientes=("CustomerID", "size"), receita=("monetario", "sum"),
                           pct_alto_risco=("risco", lambda s: round((s == "Alto").mean() * 100, 1)))
                      .round(0).reset_index().to_dict(orient="records"))
    return _json({"dashboard": "exibido na tela", "filtro": titulo, "clientes": n, "receita_total": round(receita, 2),
                  "pct_clientes_alto_risco": round(len(alto) / n * 100, 1),
                  "receita_em_risco": round(alto["monetario"].sum(), 2),
                  "segmentos": resumo_seg, "top5_clientes_retencao": top["CustomerID"].head(5).tolist()})

print("✅ Ferramenta criar_dashboard pronta")

✅ Ferramenta criar_dashboard pronta


In [ ]:
#@title 9. Atualizar o agente com a nova ferramenta
FERRAMENTAS = [f for f in FERRAMENTAS if f.__name__ != "criar_dashboard"] + [criar_dashboard]

config = types.GenerateContentConfig(
    system_instruction=INSTRUCOES + """
- Quando pedirem dashboard, painel ou gráficos, use a ferramenta criar_dashboard.
  Depois que ele aparecer, comente em tópicos os 3 principais destaques e o que o gestor deve fazer.""",
    tools=FERRAMENTAS,
    temperature=0.2,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=10),
)
novo_chat()
print("Agente atualizado ✅ Ferramentas:", [f.__name__ for f in FERRAMENTAS])

Agente atualizado ✅ Ferramentas: ['resumo_da_base', 'vendas_por_mes', 'ranking', 'segmentar_clientes', 'prever_churn', 'clientes_em_risco', 'exportar_para_power_bi', 'criar_dashboard']


In [ ]:
perguntar("Monte um dashboard com os dados dos clientes e me diga o que mais chama atenção.")

🧑 Monte um dashboard com os dados dos clientes e me diga o que mais chama atenção.

   🔧 Agente usou a ferramenta: criar_dashboard(segmento='todos')


O dashboard foi gerado com sucesso. Aqui estão os três principais destaques da análise:

*   **Concentração de Receita:** Temos um grupo de "Alto Valor" que, embora represente cerca de 38% da base de clientes, é responsável pela vasta maioria do faturamento (£ 7,5 milhões de um total de £ 8,9 milhões).
*   **Risco de Churn no Segmento de Baixo Valor:** O segmento de "Baixo Valor" apresenta uma taxa de risco de churn significativamente maior (17%) em comparação ao segmento de "Alto Valor" (apenas 1%). Isso indica que estamos perdendo clientes que já compram pouco, mas que poderiam ser nutridos.
*   **Impacto Financeiro do Churn:** Existe um montante de aproximadamente £ 390 mil em receita anual que está em risco imediato devido à probabilidade de churn dos clientes atuais.

### Recomendações para o Gestor:

1.  **Programa de Fidelidade para o Segmento de Alto Valor:** Como o risco de churn neste grupo é baixíssimo (1%), o foco deve ser em estratégias de *upsell* e *cross-sell* para aumentar ainda mais o ticket médio, garantindo que eles se sintam valorizados e continuem fiéis.
2.  **Campanha de Reengajamento para o Segmento de Baixo Valor:** Com 17% de risco de churn, precisamos de ações de marketing direto (e-mail marketing, cupons de desconto ou ofertas personalizadas) para reativar esses clientes antes que eles parem de comprar definitivamente.
3.  **Priorização de Contato:** Utilize a lista dos 5 principais clientes prioritários para retenção (exibida no dashboard) para um contato comercial direto ou um atendimento personalizado, visando garantir a permanência desses perfis estratégicos.

## Como isso se conecta ao TCC
- **Seção 1.1 (IA):** o agente segue a definição de Russell e Norvig: percebe, decide e age.
- **Seção 1.4 (análise aumentada):** é exatamente a ideia de usar IA e linguagem natural para automatizar a análise e torná-la acessível a quem não programa (PRAT, 2019).
- **Estudo de caso:** as ferramentas implementam as fases do CRISP-DM (preparação, modelagem com k-means e árvore de decisão, avaliação e implantação no Power BI).

**Limitações para citar no trabalho:** o modelo de linguagem pode errar na interpretação, então os números devem sempre vir das ferramentas, e as respostas variam um pouco a cada execução. Usar a API também envia os resultados agregados para um serviço externo (Google).